In [ ]:
import pandas as pd
%load_ext autoreload
%autoreload 2

from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import napari
import colorcet as cc
from tqdm import tqdm

import dnt

spots_directory = Path(r"C:\Tracking\BlastodermAnalysis\data\spots")
embryo_overview = pd.read_excel(spots_directory / "overview.xlsx", sheet_name="Sheet1")
save_path = Path(r"C:\Tracking\BlastodermAnalysis\figures\figure_4")


embryo_overview = embryo_overview[embryo_overview["good"]]
included = embryo_overview["Embryo"].astype(str).tolist()
condition_map = {
    str(embryo): condition for embryo, condition in zip(embryo_overview["Embryo"], embryo_overview["condition"])
}
print(condition_map)

dnt.set_plot_style()
spots_dfs, stems = dnt.load_spots_data(spots_directory, included)

print(stems)

df = spots_dfs[0]
cycles = [10, 11, 12, 13, 14]

print(df.columns)

greens = ["#143601","#1a4301","#245501","#538d22","#73a942","#aad576"][::-1]
blues = ["#012a4a","#01497c","#2a6f97","#468faf","#89c2d9"][::-1]
reds = ["#641220","#85182a","#a71e34","#bd1f36", "#da1e37"][::-1]
oranges = ["#fbba72","#ca5310","#bb4d00","#8f250c","#691e06"]
condition_pal_map = {
    "wt": blues,
    "bcd": reds,
    "trk": greens,
}
condition_main_colors = {
    c: cmap[2] for c, cmap in condition_pal_map.items()
}

condition_cycle_map = {}
for condition in condition_pal_map:
    for cycle in range(10, 15):
        condition_cycle_map[f"{condition}-{cycle}"] = condition_pal_map[condition][cycle-10]

all_mmfs = {}

for k in range(len(spots_dfs)):
    stem = stems[k]
    df = spots_dfs[k]

    min_mvmt_frames, times = dnt.find_stationary_timepoints(df)
    all_mmfs[stem] = min_mvmt_frames

In [ ]:
k = 0

df = spots_dfs[k]
stem = stems[k]
print(stem)
print(all_mmfs[stems[k]])

for frame in all_mmfs[stem]:

    frame_df = df.query("frame == @frame")

    points = frame_df[["x", "y", "z"]].values

    mesh = dnt.calculate_density.mesh_from_points(points)

    vertex_areas = dnt.calculate_density.vertex_voronoi_areas(mesh.vertices, mesh.faces)

    # print(np.sum(vertex_areas))
    #
    # print(pd.Series(vertex_areas).describe())

df["surface_area"] = np.nan
df["fraction_surface_area"] = np.nan

for frame, group in tqdm(df.groupby("frame")):

    points = group[["x", "y", "z"]].values

    mesh = dnt.calculate_density.mesh_from_points(points)
    vertex_areas = dnt.calculate_density.vertex_voronoi_areas(mesh.vertices, mesh.faces)

    df.loc[group.index, "surface_area"] = vertex_areas
    df.loc[group.index, "fraction_surface_area"] = vertex_areas / vertex_areas.sum()


In [ ]:
sns.histplot(df, x="fraction_surface_area", bins=50)

In [ ]:
import napari

viewer = napari.Viewer()

cmap = sns.color_palette("viridis", as_cmap=True)

# get colors from fraction surface area
colors = [cmap(fraction * 250) for fraction in df["fraction_surface_area"]]
viewer.add_points(df[["frame", "z", "y", "x"]].values, face_color=colors, size=5)

In [ ]:
region_colors = np.array(["#0a9396", "#ee9b00", "#ae2012"])
regions = ["Anterior", "Middle", "Posterior"]

k = 0
df = spots_dfs[k]

t = df.groupby(["track_id", "frame"])[["x", "y", "z", "time_since_nc11", "AP", "theta", "cycle"]].mean().reset_index()

t["AP_from_start"] = t["AP"] - t["track_id"].map(t.groupby("track_id")["AP"].first())
t["AP_from_start_percent"] = t["AP_from_start"] * 100
t["time_since_nc11"] = np.round(t["time_since_nc11"], 3)

fig, axes = plt.subplots(figsize=(4.5, 2.2))

for i, ap_group in enumerate([(0.05, 0.25), (0.4, 0.6), (0.75, 0.95)]):

    region_track_ids = t[t["AP"].between(*ap_group)]["track_id"].unique()
    early_track_ids = t.groupby("track_id")["time_since_nc11"].min() < 0
    early_track_ids = t["track_id"].unique()[early_track_ids]

    all_good = np.intersect1d(region_track_ids, early_track_ids)
    region_df = df.query("track_id in @all_good").copy()

    area = region_df.groupby("time_since_nc11")["surface_area"].sum()
    area_normed = area / area.iloc[0]

    sns.lineplot(area_normed, color=region_colors[i], errorbar=None, alpha=1, label=regions[i], linewidth=4, legend=False)

    for cycle in cycles[1:]:
        cycle_df = t[t["cycle"] == cycle]
        cycle_times = cycle_df.groupby("track_id")["time_since_nc11"].min()
        division_time = cycle_times.median()
        plt.axvline(division_time, color="k", linestyle="--", linewidth=2, alpha=0.2)

    axes.spines["top"].set_visible(False)
    axes.spines["right"].set_visible(False)

plt.title(f"Area change of region over time")
plt.ylabel("Region Area (fold change)")
plt.xlabel("Time since nc11 (minutes)")
# plt.ylim(-0.5, 1.5)
# plt.legend()
plt.savefig(save_path / f"{stems[k]}_area_change_over_time.png", dpi=300, bbox_inches="tight")
plt.show()

# make bands graphic with new data

In [ ]:
print(all_mmfs[stems[7]])

In [ ]:
import pandas as pd
from tqdm import tqdm
from collections import defaultdict


df = spots_dfs[7]

mesh_path = save_path / "all_meshes_trk"
mesh_path.mkdir(exist_ok=True)

def hex2rgb(color):
    return int(color[1:3], 16) / 255, int(color[3:5], 16) / 255, int(color[5:7], 16) / 255


base_color = r"#e9d8a6"
r, g, b = hex2rgb(base_color)

colors = [hex2rgb(c) for c in
          ["#0a9396", "#ee9b00", "#ae2012"]]

track_id_colors = {}

for i, region in enumerate([(0.05, 0.25), (0.4, 0.6), (0.75, 0.95)]):
    track_ids = df.query("frame < 30 and AP.between(@region[0], @region[1])")["track_id"].unique()

    for track_id in track_ids:
        track_id_colors[track_id] = colors[i]

for i, frame in tqdm(enumerate(df["frame"].unique())):

    frame_df = df.query("frame == @frame")

    points = frame_df[["z", "y", "x"]].values
    mesh = dnt.mesh_from_points(points)

    mesh.write_obj(mesh_path / f"frame_{frame}.obj")
    track_ids = frame_df["track_id"].values

    colors = [track_id_colors.get(tid, hex2rgb(base_color)) for tid in track_ids]

    valid = pd.Series(np.arange(len(points))).isin(np.unique(mesh.faces))
    blender_save = pd.DataFrame(np.array(colors)[valid], columns=["R", "G", "B"])
    blender_save["track_id"] = track_ids[valid]

    blender_save.to_csv(mesh_path / f"frame_{frame}_colors.csv", index=False)